## MongoDB Code Inconsistency Database Builder

This notebook will build the code inconsistency database. It uses the code generation database, so do build the code generation database first before building this database.

The new database will contain processed data in a new schema used specifically for code inconsistency testing. 

In [1]:
import os
import sys
from tqdm import tqdm
import pandas as pd

In [2]:
curr_dir = os.getcwd()
par_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(par_dir)
sys.path.append(proj_dir)

In [3]:
from database import MongoDBHelper
from mcq_inconsistency.utility.codemmlu_helper import CodeGenerationCodeMMLUHelper

In [4]:
db = MongoDBHelper()
if db.check_database_connectivity():
    print("MongoDB connected")

MongoDB connected


In [5]:
base_qns_db = db.client["Base_Questions_DB"]
codemmlu_database = pd.read_csv(
    "/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/datasets/open_ended_format/codemmlu_test.csv",
    encoding="utf-8",
    header=0,
    )

humaneval_database = pd.read_csv(
    "/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/datasets/open_ended_format/humaneval_test_modified_open.csv",
    encoding="utf-8",
    header=0,
    quoting=1,           
    )

In [6]:
codemmlu_task = "code_completion"
mcq_question_database = base_qns_db[f"CodeMMLU_MCQ_{codemmlu_task}"]

In [7]:
ans_dict = {
    "A" : 0,
    "B" : 1,
    "C" : 2,
    "D" : 3,
}

failed_to_upload = []

for i in tqdm(range(
    len(codemmlu_database)
    # 116, 117
    # 1
    )):
    try:
        input_task_id = f"CodeMMLU{i}"

        codemmlu_qn = codemmlu_database.iloc[i]
        question = codemmlu_qn['question']
        choices = codemmlu_qn['choices']
        expected_ans = codemmlu_qn['answer']
        original_id = codemmlu_qn['task_id']

        humaneval_qn = humaneval_database.iloc[i]
        test_suite = humaneval_qn['test']
        func_name = humaneval_qn['entry_point']
        humaneval_id = humaneval_qn['task_id']

        if isinstance(choices, str):
            choices = eval(choices)

        question, qn_desc = CodeGenerationCodeMMLUHelper.seperate_original_desciptions(question)
        qn_desc, examples = CodeGenerationCodeMMLUHelper.extract_examples(qn_desc)
        choices = [CodeGenerationCodeMMLUHelper._standardize_leading_whitespaces(choice) for choice in choices ]

        correct_choice = choices[ans_dict[expected_ans]]

        full_sol = question + "\n" + correct_choice

        #sanity check for codemmlu full solution
        test_suite = CodeGenerationCodeMMLUHelper.process_original_tests(test_suite)

        validate_full_sol = CodeGenerationCodeMMLUHelper.check_test_case(
            test_case = test_suite,
            code_snippet = full_sol,
            func_name = func_name
        )

        if not validate_full_sol:
            failed_to_upload.append(original_id)
            raise ValueError("Full Solution Failed the test suite")

        database_entry = {
            "_id": input_task_id,
            "question": question,
            "qn_desc": qn_desc,
            "choices": choices,
            "check": test_suite,
            "answer": expected_ans,
            "examples": examples,
            "func_name": func_name,
            "original_id": original_id,
            "corresponding_humaneval_id": humaneval_id
        }

        exisiting_entry = mcq_question_database.find_one({"_id": input_task_id})

        if exisiting_entry is not None:
            mcq_question_database.find_one_and_replace({"_id": input_task_id}, database_entry)
        else:
            mcq_question_database.insert_one(database_entry)
        
        ## Next, we need to ensure that the entry stored in the DB
        db_entry = mcq_question_database.find_one(filter = {"_id": input_task_id})
        
        question = db_entry['question']
        choices = db_entry['choices']
        check = db_entry['check']
        answer = db_entry['answer']
        func_name = db_entry['func_name']

        correct_choice = choices[ans_dict[answer]]

        full_sol = question + '\n' + correct_choice

        validate_full_sol = CodeGenerationCodeMMLUHelper.check_test_case(
            test_case = test_suite,
            code_snippet = full_sol,
            func_name = func_name
        )

        if validate_full_sol is not True:
            mcq_question_database.find_one_and_delete({"_id": input_task_id})
            failed_to_upload.append(original_id)
    except Exception as e:
        print(original_id, f"failed to upload into the database due to following error: {e}")
    

  0%|          | 0/164 [00:00<?, ?it/s]

100%|██████████| 164/164 [00:04<00:00, 35.35it/s]


In [8]:
if len(failed_to_upload) < 1:
    print('All tasks uploaded successfully!')
else:
    print(f"The following tasks failed: {failed_to_upload}")

All tasks uploaded successfully!


* Modified rt00012 answer from B to D. Original answer (B) was incorrect and D is correct.
* Modified rt00052 examples by removing ">>> remove_vowels("abcdef\nghijklm")" example. The \n messes up the process
* rt00067 to rt00164 examples are modified to doctest format